# 실제 Agent HTTP 통합

OpenAI 모델과 조회 도구가 있는 LangChain `create_agent`를 FastAPI에서 실제 호출합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

### 이 준비 셀이 하는 일

이 준비 셀은 `Path.cwd().resolve()`로 현재 폴더를 절대 경로로 읽고, 현재 폴더와 부모 폴더에서 `agent_service`가 있는 첫 위치를 `next(..., None)`로 찾습니다. 찾지 못하면 `assert`가 실행을 중단합니다. 찾은 뒤 `os.chdir(lab)`이 이후 import와 파일 접근의 기준 폴더를 바꾸고, 마지막 출력은 실제 선택된 폴더 이름을 확인합니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 키 존재 확인

상위 폴더의 `.env`를 읽고 키 값은 출력하지 않습니다.

### API 연결에 필요한 환경 변수 확인

`find_dotenv(usecwd=True)`는 현재 폴더부터 상위 폴더에서 `.env`를 찾고, `load_dotenv()`는 환경 변수를 읽습니다. `os.getenv()`로 키 설정 여부를 읽고 `bool()`로 참·거짓만 출력합니다. `assert has_key`는 키가 없으면 중단합니다. 이 셀은 모델이나 HTTP 경로를 호출하지 않습니다.


In [2]:
import os
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(usecwd=True))
has_key = bool(os.getenv('OPENAI_API_KEY'))
print('OPENAI_API_KEY 설정:', has_key)
assert has_key

OPENAI_API_KEY 설정: True


## 실제 ainvoke HTTP 요청

Agent는 운영 시간을 도구로 확인한 뒤 한 문장으로 답합니다.

`TestClient(app)`는 같은 프로세스 안에서 FastAPI 요청 처리를 시험하는 클라이언트입니다. `post()`에 경로, 인증 헤더, JSON 본문을 넘기면 해당 경로가 실행됩니다. `/v1/chat/live`의 내부 Agent는 실제 모델을 호출하므로 외부 모델 API 연결이 필요합니다. 반환된 HTTP 상태 코드와 답 문자열을 함께 확인합니다.


In [3]:
from fastapi.testclient import TestClient
from agent_service.app import app
client = TestClient(app)
headers = {'Authorization': 'Bearer local-demo-token'}
body = {'question': '한빛도서관 월요일 운영 시간은?', 'thread_id': 'live-1'}
response = client.post('/v1/chat/live', headers=headers, json=body)
print(response.status_code, response.json()['answer'])
assert response.status_code == 200
assert '18:00' in response.json()['answer']

200 한빛도서관 월요일 운영 시간은 09:00~18:00입니다.


## 도구 호출 궤적 확인

같은 실제 Agent의 메시지에서 도구 이름, 입력, 도구 출력을 확인합니다.

`build_live_agent()`는 호출 객체를 준비하고 `await live_agent.ainvoke(...)`가 별도의 실제 실행을 시작합니다. `tool_calls`에는 모델이 요청한 도구 이름과 인수가 있고, `ToolMessage`에는 실행한 도구의 결과가 있습니다. 앞 HTTP 호출의 저장 궤적을 읽는 셀이 아니라 같은 질문으로 별도 실행한 궤적을 확인하는 셀입니다.


In [4]:
from langchain_core.messages import ToolMessage
from agent_service.live_agent import build_live_agent
live_agent = build_live_agent()
trace = await live_agent.ainvoke({'messages': [
    {'role': 'user', 'content': body['question']}
]})
tool_call = next(call for message in trace['messages']
    for call in getattr(message, 'tool_calls', [])
    if call['name'] == 'lookup_library_hours')
tool_result = next(message for message in trace['messages']
    if isinstance(message, ToolMessage))
print(tool_call['name'], tool_call['args'], tool_result.content)
assert tool_call['args']['day'] == '월요일'
assert tool_result.content == '09:00~18:00'

lookup_library_hours {'day': '월요일'} 09:00~18:00


## 실제 astream HTTP 요청

같은 Agent의 모델 메시지 청크를 SSE token 이벤트로 받습니다.

SSE는 이벤트 이름과 데이터를 텍스트 블록으로 보내는 형식입니다. 아래에서는 `event: token` 블록을 고르고 `data:` 뒤의 JSON을 읽어 `token` 문자열을 연결합니다. TestClient에서 완성된 응답을 받아 해석하므로 화면에 조각이 실시간으로 보이는 지연을 측정하지 않습니다.


In [5]:
import json, re
stream = client.post('/v1/chat/live/stream', headers=headers, json=body)
blocks = [b for b in re.split(r'\r?\n\r?\n', stream.text) if b]
tokens = []
for block in blocks:
    if block.startswith('event: token'):
        data_line = next(line for line in block.splitlines() if line.startswith('data: '))
        tokens.append(json.loads(data_line[6:])['token'])
live_answer = ''.join(tokens)
print(stream.status_code, live_answer)
assert stream.status_code == 200
assert blocks[-1].startswith('event: done')
assert '18:00' in live_answer

200 한빛도서관 월요일 운영 시간은 09:00~18:00입니다.


## 실험 결과 해석

결정적 그래프와 실제 모델 Agent의 출력 성질을 구분합니다.

### 이 단계에서 구분할 범위

답에 `18:00`이 포함됐다는 한 조건만으로 올바른 도구 실행을 증명할 수 없습니다. 모델이 우연히 같은 숫자를 생성했을 수도 있기 때문입니다. 그래서 HTTP 응답, 도구 호출 인수, 도구 결과, 스트림 완료를 서로 다른 관찰로 남깁니다. 이 검사는 현재 질문 한 건의 통합 경로를 확인하며 다른 요일과 오류 입력의 품질까지 일반화하지 않습니다.

In [6]:
token_events = stream.text.count('event: token')
print('실제 모델 token 이벤트:', token_events)
print('결정적 그래프가 아닌 create_agent 호출: True')
assert token_events > 0

실제 모델 token 이벤트: 20
결정적 그래프가 아닌 create_agent 호출: True
